# Web Scraping Yelp Reviews with Beautiful Soup

This notebook demonstrates how to scrape reviews from Yelp using Beautiful Soup and the requests library.

## 1. Import Required Libraries

Import necessary libraries including requests, BeautifulSoup, pandas, and time.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

## 2. Set Up HTTP Headers and Session

Configure HTTP headers to mimic a browser request. This helps avoid being blocked by the server.

In [ ]:
# Set up headers to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1'
}

# Create a session for persistent connection
session = requests.Session()
session.headers.update(headers)

## 3. Define Target URL and Parameters

Specify the Yelp business URL to scrape reviews from. 

**Note:** Yelp's website is heavily JavaScript-based, so Beautiful Soup alone may not retrieve all content. For production use, consider using Selenium or the official Yelp Fusion API.

In [ ]:
# Example: Target a specific restaurant page
# Replace this with an actual Yelp business URL
business_url = "https://www.yelp.com/biz/restaurant-name-city"

# For searching restaurants in a specific location
search_url = "https://www.yelp.com/search?find_desc=Restaurants&find_loc=55403"

print(f"Target URL: {search_url}")

## 4. Fetch Web Page Content

Use requests to fetch the HTML content from the Yelp page.

In [ ]:
def fetch_page_content(url):
    """
    Fetch HTML content from a URL
    
    Args:
        url (str): The URL to fetch
        
    Returns:
        str: HTML content or None if failed
    """
    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()  # Raise error for bad status codes
        print(f"✓ Successfully fetched page (Status: {response.status_code})")
        print(f"  Content length: {len(response.content)} bytes")
        return response.content
    except requests.exceptions.RequestException as e:
        print(f"✗ Error fetching page: {e}")
        return None

# Test fetching a page
html_content = fetch_page_content(search_url)

if html_content:
    # Check if we're getting actual content or being blocked
    if b"enable JS" in html_content or b"captcha" in html_content.lower():
        print("⚠️  WARNING: Yelp is requiring JavaScript or showing a CAPTCHA")
        print("   Beautiful Soup alone won't work. Consider using Selenium instead.")

## 5. Parse HTML with Beautiful Soup

Parse the HTML content using Beautiful Soup to create a navigable tree structure.

In [ ]:
def parse_html(html_content):
    """
    Parse HTML content with Beautiful Soup
    
    Args:
        html_content: Raw HTML content
        
    Returns:
        BeautifulSoup object
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    return soup

# Parse the fetched content
if html_content:
    soup = parse_html(html_content)
    
    # Display some basic info about the page
    title = soup.title.string if soup.title else "No title found"
    print(f"Page title: {title}")
    
    # Save HTML for inspection
    with open('yelp_page_inspect.html', 'w', encoding='utf-8') as f:
        f.write(soup.prettify())
    print("✓ Saved HTML to 'yelp_page_inspect.html' for inspection")

## 6. Extract Review Data

Identify and extract review elements including rating, text, date, and user information using CSS selectors.

**Note:** These selectors are approximations. Yelp frequently changes its HTML structure and requires JavaScript to load reviews.

In [ ]:
def extract_reviews(soup):
    """
    Extract review data from parsed HTML
    
    Args:
        soup: BeautifulSoup object
        
    Returns:
        list: List of dictionaries containing review data
    """
    reviews = []
    
    # Try multiple selectors as Yelp's structure varies
    # Look for review containers
    review_containers = soup.find_all('li', {'class': lambda x: x and 'review' in str(x).lower()})
    
    if not review_containers:
        review_containers = soup.find_all('div', {'data-review-id': True})
    
    if not review_containers:
        # Try to find any elements with lang attribute (often review text)
        review_containers = soup.find_all(['div', 'section', 'article'], recursive=True)
    
    print(f"Found {len(review_containers)} potential review containers")
    
    for i, container in enumerate(review_containers):
        try:
            # Extract review text
            review_text = None
            text_elem = container.find('p', {'class': lambda x: x and 'comment' in str(x).lower()})
            
            if not text_elem:
                text_elem = container.find('span', {'lang': True})
            
            if not text_elem:
                text_elem = container.find('p')
            
            if text_elem:
                review_text = text_elem.get_text(strip=True)
            
            # Extract rating
            rating = None
            rating_elem = container.find(['div', 'span'], {'aria-label': lambda x: x and 'star' in str(x).lower()})
            
            if rating_elem:
                aria_label = rating_elem.get('aria-label', '')
                if 'star' in aria_label.lower():
                    rating = aria_label.split()[0]
            
            # Extract date (if available)
            date_elem = container.find('span', {'class': lambda x: x and 'date' in str(x).lower()})
            date = date_elem.get_text(strip=True) if date_elem else None
            
            # Extract user name (if available)
            user_elem = container.find('a', {'class': lambda x: x and 'user' in str(x).lower()})
            if not user_elem:
                user_elem = container.find('span', {'class': lambda x: x and 'user' in str(x).lower()})
            user_name = user_elem.get_text(strip=True) if user_elem else None
            
            # Only add reviews with substantial text
            if review_text and len(review_text) > 30:
                reviews.append({
                    'review_text': review_text,
                    'rating': rating if rating else 'N/A',
                    'date': date,
                    'user': user_name,
                    'scraped_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                })
                
        except Exception as e:
            print(f"Error extracting review {i}: {e}")
            continue
    
    print(f"✓ Successfully extracted {len(reviews)} reviews")
    return reviews

# Extract reviews from the parsed HTML
if html_content and soup:
    reviews_data = extract_reviews(soup)
    
    if reviews_data:
        print("\nSample review:")
        print(f"Text: {reviews_data[0]['review_text'][:100]}...")
        print(f"Rating: {reviews_data[0]['rating']}")
    else:
        print("\n⚠️  No reviews extracted. This is expected with Yelp's JavaScript-heavy pages.")
        print("   Consider using Selenium for JavaScript-rendered content.")

## 7. Store Reviews in DataFrame

Create a pandas DataFrame to store the extracted review data in a structured format.

In [ ]:
def create_reviews_dataframe(reviews_data):
    """
    Create a pandas DataFrame from reviews data
    
    Args:
        reviews_data (list): List of review dictionaries
        
    Returns:
        DataFrame: Pandas DataFrame with review data
    """
    if not reviews_data:
        print("No reviews to create DataFrame")
        return pd.DataFrame()
    
    df = pd.DataFrame(reviews_data)
    
    # Display basic info
    print(f"DataFrame created with {len(df)} reviews")
    print(f"Columns: {list(df.columns)}")
    
    return df

# Create DataFrame
if 'reviews_data' in locals() and reviews_data:
    reviews_df = create_reviews_dataframe(reviews_data)
    
    # Display first few reviews
    if not reviews_df.empty:
        print("\nFirst 3 reviews:")
        print(reviews_df.head(3))
        
        # Basic statistics
        print(f"\nTotal reviews: {len(reviews_df)}")
        if 'rating' in reviews_df.columns:
            print(f"Ratings distribution:\n{reviews_df['rating'].value_counts()}")
else:
    reviews_df = pd.DataFrame()
    print("⚠️  Creating empty DataFrame - no reviews were extracted")

## 8. Export Data to CSV

Save the scraped reviews to a CSV file for further analysis.

In [ ]:
def save_to_csv(df, filename='yelp_reviews.csv'):
    """
    Save DataFrame to CSV file
    
    Args:
        df: Pandas DataFrame
        filename: Output filename
    """
    if df.empty:
        print("⚠️  DataFrame is empty, not saving to CSV")
        return
    
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"✓ Saved {len(df)} reviews to '{filename}'")

# Save to CSV
if 'reviews_df' in locals() and not reviews_df.empty:
    save_to_csv(reviews_df, 'yelp_reviews_beautifulsoup.csv')
else:
    print("No data to save")

---

## Alternative: Complete Web Scraping Function

Here's a complete function that combines all the steps above:

In [ ]:
def scrape_yelp_reviews(url, output_file='yelp_reviews.csv'):
    """
    Complete function to scrape Yelp reviews
    
    Args:
        url: Yelp search or business page URL
        output_file: CSV filename for output
        
    Returns:
        DataFrame: Reviews data
    """
    print(f"Starting scrape of: {url}\n")
    
    # Fetch page
    html = fetch_page_content(url)
    if not html:
        return pd.DataFrame()
    
    # Parse HTML
    soup = parse_html(html)
    
    # Extract reviews
    reviews = extract_reviews(soup)
    
    # Create DataFrame
    df = create_reviews_dataframe(reviews)
    
    # Save to CSV
    if not df.empty:
        save_to_csv(df, output_file)
    
    return df

# Example usage:
# result_df = scrape_yelp_reviews(search_url, 'my_yelp_reviews.csv')

---

## Important Notes About Yelp Scraping

### Why Beautiful Soup Doesn't Work Well for Yelp:

1. **JavaScript Rendering**: Yelp loads most content dynamically with JavaScript. Beautiful Soup only sees the initial HTML, not the JavaScript-rendered content.

2. **Anti-Scraping Measures**: Yelp actively blocks scrapers with CAPTCHAs and bot detection.

3. **Terms of Service**: Scraping Yelp may violate their Terms of Service.

### Better Alternatives:

1. **Yelp Fusion API** (Recommended):
   - Official, legal way to access Yelp data
   - Free tier available
   - https://www.yelp.com/developers

2. **Selenium WebDriver**:
   - Can execute JavaScript
   - Mimics real browser behavior
   - More complex but more effective

3. **Use Existing Datasets**:
   - Yelp Open Dataset: https://www.yelp.com/dataset
   - Pre-collected and legal to use for research

---

## Selenium Alternative (For Reference)

If you need to scrape JavaScript-heavy sites like Yelp, here's how to use Selenium:

```python
# First install: pip install selenium webdriver-manager

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Setup driver
options = webdriver.ChromeOptions()
options.add_argument('--headless')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load page
driver.get(url)
time.sleep(3)  # Wait for JavaScript to load

# Extract data
reviews = driver.find_elements(By.CSS_SELECTOR, 'p[lang]')
for review in reviews:
    print(review.text)

driver.quit()
```